
# 05 — Training

This notebook trains the CNN Encoder → LSTM Decoder baseline defined in `04_baseline_model_corrected.ipynb`.

The notebook intentionally keeps the same project conventions:

- CROHME 2019 processed manifests and rendered grayscale images
- canonical vocabulary from `data/processed/crohme2019/metadata/vocab.json`
- `<PAD>=0`, `<SOS>=1`, `<EOS>=2`, `<UNK>=3`
- dynamic image padding in the batch
- token-level cross-entropy with padding ignored
- no attention
- CNN encoder: `1 → 32 → 64 → 128 → 256 → 512`
- LSTM decoder: embedding `256`, hidden size `512`, one layer
- token-level ExpRate for validation/checkpoint selection


In [ ]:

from pathlib import Path
import json
import random
import time
import math
import os

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt


In [ ]:

# ---------------------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Keep CUDA deterministic when possible.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("Seed:", SEED)


In [ ]:

# ---------------------------------------------------------------------
# Project paths
# ---------------------------------------------------------------------

PROJECT_ROOT = Path("/data1/students/abhista/HMER")

PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed" / "crohme2019"

IMAGE_ROOT = PROCESSED_ROOT / "images"
MANIFEST_ROOT = PROCESSED_ROOT / "manifests"
METADATA_ROOT = PROCESSED_ROOT / "metadata"

OUTPUT_ROOT = PROJECT_ROOT / "outputs"
MODEL_ROOT = OUTPUT_ROOT / "models"
TRAINING_ROOT = OUTPUT_ROOT / "training"

MODEL_ROOT.mkdir(parents=True, exist_ok=True)
TRAINING_ROOT.mkdir(parents=True, exist_ok=True)

TRAIN_MANIFEST = MANIFEST_ROOT / "train.jsonl"
VALID_MANIFEST = MANIFEST_ROOT / "valid.jsonl"
TEST_MANIFEST = MANIFEST_ROOT / "test.jsonl"

VOCAB_PATH = METADATA_ROOT / "vocab.json"

print("Project root :", PROJECT_ROOT)
print("Processed    :", PROCESSED_ROOT)
print("Model output :", MODEL_ROOT)
print("Training out :", TRAINING_ROOT)


In [ ]:

# ---------------------------------------------------------------------
# Training configuration
# ---------------------------------------------------------------------

BATCH_SIZE = 32
NUM_EPOCHS = 20
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5

EMBEDDING_DIM = 256
HIDDEN_DIM = 512
NUM_LAYERS = 1

MAX_TARGET_LENGTH = 150

NUM_WORKERS = 0
PIN_MEMORY = torch.cuda.is_available()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CHECKPOINT_PATH = MODEL_ROOT / "baseline_best.pt"
LAST_CHECKPOINT_PATH = MODEL_ROOT / "baseline_last.pt"
HISTORY_PATH = TRAINING_ROOT / "baseline_training_history.json"

print("Device:", DEVICE)
print("Batch size:", BATCH_SIZE)
print("Epochs:", NUM_EPOCHS)
print("Learning rate:", LEARNING_RATE)


In [ ]:

# ---------------------------------------------------------------------
# Validate required files before training
# ---------------------------------------------------------------------

required_paths = [
    TRAIN_MANIFEST,
    VALID_MANIFEST,
    TEST_MANIFEST,
    VOCAB_PATH,
]

for path in required_paths:
    if not path.exists():
        raise FileNotFoundError(f"Required file not found: {path}")

print("All required metadata files are present.")


In [ ]:

# ---------------------------------------------------------------------
# Load vocabulary
# ---------------------------------------------------------------------

with open(VOCAB_PATH, "r", encoding="utf-8") as f:
    vocab = json.load(f)

# The canonical vocabulary generated during preprocessing is expected
# to contain token -> integer mappings.
if "token_to_id" in vocab:
    token_to_id = vocab["token_to_id"]
    id_to_token_raw = vocab.get("id_to_token")
elif "stoi" in vocab:
    token_to_id = vocab["stoi"]
    id_to_token_raw = vocab.get("itos")
else:
    # Also support a plain token -> id dictionary.
    token_to_id = vocab
    id_to_token_raw = None

token_to_id = {str(k): int(v) for k, v in token_to_id.items()}

if id_to_token_raw is not None:
    if isinstance(id_to_token_raw, list):
        id_to_token = {i: token for i, token in enumerate(id_to_token_raw)}
    else:
        id_to_token = {int(k): str(v) for k, v in id_to_token_raw.items()}
else:
    id_to_token = {v: k for k, v in token_to_id.items()}

PAD_ID = token_to_id["<PAD>"]
SOS_ID = token_to_id["<SOS>"]
EOS_ID = token_to_id["<EOS>"]
UNK_ID = token_to_id["<UNK>"]

VOCAB_SIZE = len(token_to_id)

print("Vocabulary size:", VOCAB_SIZE)
print("PAD:", PAD_ID)
print("SOS:", SOS_ID)
print("EOS:", EOS_ID)
print("UNK:", UNK_ID)


In [ ]:

# ---------------------------------------------------------------------
# Manifest loader
# ---------------------------------------------------------------------

def load_manifest(path):
    records = []

    with open(path, "r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            line = line.strip()

            if not line:
                continue

            try:
                record = json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(
                    f"Invalid JSON in {path} at line {line_number}"
                ) from exc

            records.append(record)

    return records


train_records = load_manifest(TRAIN_MANIFEST)
valid_records = load_manifest(VALID_MANIFEST)
test_records = load_manifest(TEST_MANIFEST)

print(f"Train records: {len(train_records)}")
print(f"Valid records: {len(valid_records)}")
print(f"Test records : {len(test_records)}")


In [ ]:

# ---------------------------------------------------------------------
# Image-path resolver
#
# The manifests store relative image paths. This resolver accepts the
# processed-image layout produced by notebook 02 and also handles the
# common case where the manifest path already contains "images/".
# ---------------------------------------------------------------------

def resolve_image_path(relative_path):
    relative_path = Path(str(relative_path))

    candidates = [
        PROCESSED_ROOT / relative_path,
        IMAGE_ROOT / relative_path,
        PROJECT_ROOT / relative_path,
    ]

    for candidate in candidates:
        if candidate.exists():
            return candidate

    raise FileNotFoundError(
        "Could not resolve image path. "
        f"Manifest value: {relative_path}"
    )


In [ ]:

# ---------------------------------------------------------------------
# Dataset
# ---------------------------------------------------------------------

class CROHMEDataset(Dataset):
    def __init__(self, records, max_target_length=150):
        self.records = records
        self.max_target_length = max_target_length

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        record = self.records[index]

        image_path = resolve_image_path(record["image"])

        image = Image.open(image_path).convert("L")
        image = np.asarray(image, dtype=np.float32) / 255.0

        image = torch.from_numpy(image).unsqueeze(0)

        if "token_ids" in record:
            token_ids = [int(x) for x in record["token_ids"]]
        elif "tokens" in record:
            token_ids = [
                token_to_id.get(str(token), UNK_ID)
                for token in record["tokens"]
            ]
        else:
            raise KeyError(
                "Manifest record must contain either 'token_ids' or 'tokens'."
            )

        # The processed manifests already contain the canonical target
        # sequence. Enforce the notebook-level maximum without removing EOS.
        if len(token_ids) > self.max_target_length:
            token_ids = token_ids[:self.max_target_length]

            if EOS_ID not in token_ids:
                token_ids[-1] = EOS_ID

        target = torch.tensor(token_ids, dtype=torch.long)

        return {
            "image": image,
            "target": target,
            "image_path": str(image_path),
            "record": record,
        }


In [ ]:

# ---------------------------------------------------------------------
# Dynamic-padding collate function
#
# Images are padded to the largest H/W in the current batch.
# Targets are padded with PAD_ID.
# ---------------------------------------------------------------------

def collate_batch(batch):
    batch_size = len(batch)

    max_h = max(item["image"].shape[1] for item in batch)
    max_w = max(item["image"].shape[2] for item in batch)

    images = torch.zeros(
        batch_size,
        1,
        max_h,
        max_w,
        dtype=torch.float32,
    )

    targets = torch.full(
        (batch_size, max(len(item["target"]) for item in batch)),
        PAD_ID,
        dtype=torch.long,
    )

    image_paths = []

    for i, item in enumerate(batch):
        image = item["image"]
        h, w = image.shape[1:]

        images[i, :, :h, :w] = image

        target = item["target"]
        targets[i, :len(target)] = target

        image_paths.append(item["image_path"])

    return {
        "images": images,
        "targets": targets,
        "image_paths": image_paths,
    }


In [ ]:

# ---------------------------------------------------------------------
# DataLoaders
# ---------------------------------------------------------------------

train_dataset = CROHMEDataset(
    train_records,
    max_target_length=MAX_TARGET_LENGTH,
)

valid_dataset = CROHMEDataset(
    valid_records,
    max_target_length=MAX_TARGET_LENGTH,
)

test_dataset = CROHMEDataset(
    test_records,
    max_target_length=MAX_TARGET_LENGTH,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    collate_fn=collate_batch,
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    collate_fn=collate_batch,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    collate_fn=collate_batch,
)

print("Train batches:", len(train_loader))
print("Valid batches:", len(valid_loader))
print("Test batches :", len(test_loader))


In [ ]:

# ---------------------------------------------------------------------
# Verify one batch
# ---------------------------------------------------------------------

sample_batch = next(iter(train_loader))

print("Images :", sample_batch["images"].shape)
print("Targets:", sample_batch["targets"].shape)
print("Image dtype :", sample_batch["images"].dtype)
print("Target dtype:", sample_batch["targets"].dtype)



## Baseline model

This is the same no-attention CNN → LSTM architecture established in notebook 04.

The CNN compresses the complete document image into one feature vector. The LSTM uses that image representation to initialize its hidden and cell states and generates the LaTeX token sequence using teacher forcing.


In [ ]:

# ---------------------------------------------------------------------
# CNN Encoder
# ---------------------------------------------------------------------

class CNNEncoder(nn.Module):
    def __init__(self, hidden_dim=HIDDEN_DIM):
        super().__init__()

        self.features = nn.Sequential(
            # 1 -> 32
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # 32 -> 64
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # 64 -> 128
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # 128 -> 256
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # 256 -> 512
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),

            nn.AdaptiveAvgPool2d((1, 1)),
        )

        self.projection = nn.Linear(512, hidden_dim)

    def forward(self, images):
        x = self.features(images)
        x = torch.flatten(x, start_dim=1)
        x = self.projection(x)
        return x


In [ ]:

# ---------------------------------------------------------------------
# LSTM Decoder
# ---------------------------------------------------------------------

class LSTMDecoder(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim=EMBEDDING_DIM,
        hidden_dim=HIDDEN_DIM,
        num_layers=NUM_LAYERS,
        pad_id=PAD_ID,
    ):
        super().__init__()

        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=pad_id,
        )

        self.image_to_hidden = nn.Linear(
            hidden_dim,
            hidden_dim * num_layers,
        )

        self.image_to_cell = nn.Linear(
            hidden_dim,
            hidden_dim * num_layers,
        )

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.0 if num_layers == 1 else 0.2,
        )

        self.output_projection = nn.Linear(
            hidden_dim,
            vocab_size,
        )

    def forward(self, image_features, target_sequences):
        batch_size = image_features.size(0)

        hidden = self.image_to_hidden(image_features)
        hidden = hidden.view(
            batch_size,
            self.num_layers,
            self.hidden_dim,
        )
        hidden = hidden.permute(1, 0, 2).contiguous()

        cell = self.image_to_cell(image_features)
        cell = cell.view(
            batch_size,
            self.num_layers,
            self.hidden_dim,
        )
        cell = cell.permute(1, 0, 2).contiguous()

        embedded = self.embedding(target_sequences)

        outputs, _ = self.lstm(
            embedded,
            (hidden, cell),
        )

        logits = self.output_projection(outputs)

        return logits


In [ ]:

# ---------------------------------------------------------------------
# Complete baseline model
# ---------------------------------------------------------------------

class BaselineHMER(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim=EMBEDDING_DIM,
        hidden_dim=HIDDEN_DIM,
        num_layers=NUM_LAYERS,
        pad_id=PAD_ID,
    ):
        super().__init__()

        self.encoder = CNNEncoder(hidden_dim=hidden_dim)

        self.decoder = LSTMDecoder(
            vocab_size=vocab_size,
            embedding_dim=embedding_dim,
            hidden_dim=hidden_dim,
            num_layers=num_layers,
            pad_id=pad_id,
        )

    def forward(self, images, target_sequences):
        image_features = self.encoder(images)
        logits = self.decoder(
            image_features,
            target_sequences,
        )
        return logits


In [ ]:

# ---------------------------------------------------------------------
# Instantiate model
# ---------------------------------------------------------------------

model = BaselineHMER(
    vocab_size=VOCAB_SIZE,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    pad_id=PAD_ID,
).to(DEVICE)

num_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print(model)
print()
print(f"Total parameters     : {num_parameters:,}")
print(f"Trainable parameters : {trainable_parameters:,}")


In [ ]:

# ---------------------------------------------------------------------
# Optimizer and loss
# ---------------------------------------------------------------------

criterion = nn.CrossEntropyLoss(
    ignore_index=PAD_ID
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

print(criterion)
print(optimizer)


In [ ]:

# ---------------------------------------------------------------------
# Token-level exact-match rate (ExpRate-style metric)
#
# A prediction is counted as correct only when the predicted token
# sequence exactly matches the ground-truth sequence after removing
# PAD tokens and terminating at EOS.
# ---------------------------------------------------------------------

def trim_sequence(sequence):
    sequence = [
        int(token)
        for token in sequence
        if int(token) != PAD_ID
    ]

    if EOS_ID in sequence:
        sequence = sequence[:sequence.index(EOS_ID) + 1]

    return sequence


def sequence_exact_match(prediction, target):
    return trim_sequence(prediction) == trim_sequence(target)


def batch_exact_match(logits, targets):
    predictions = logits.argmax(dim=-1)

    correct = 0

    for prediction, target in zip(predictions, targets):
        if sequence_exact_match(
            prediction.detach().cpu().tolist(),
            target.detach().cpu().tolist(),
        ):
            correct += 1

    return correct


In [ ]:

# ---------------------------------------------------------------------
# One training epoch
#
# Teacher forcing:
# decoder input  = target[:, :-1]
# expected output = target[:, 1:]
# ---------------------------------------------------------------------

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    total_tokens = 0

    for batch in loader:
        images = batch["images"].to(device, non_blocking=True)
        targets = batch["targets"].to(device, non_blocking=True)

        if targets.size(1) < 2:
            continue

        decoder_inputs = targets[:, :-1]
        decoder_targets = targets[:, 1:]

        optimizer.zero_grad(set_to_none=True)

        logits = model(
            images,
            decoder_inputs,
        )

        loss = criterion(
            logits.reshape(-1, VOCAB_SIZE),
            decoder_targets.reshape(-1),
        )

        loss.backward()

        # Prevent unstable gradients in the LSTM.
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=5.0,
        )

        optimizer.step()

        valid_token_count = (
            decoder_targets != PAD_ID
        ).sum().item()

        running_loss += loss.item() * max(valid_token_count, 1)
        total_tokens += valid_token_count

    epoch_loss = (
        running_loss / total_tokens
        if total_tokens > 0
        else math.nan
    )

    return epoch_loss


In [ ]:

# ---------------------------------------------------------------------
# Validation
# ---------------------------------------------------------------------

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    total_tokens = 0
    total_expressions = 0
    correct_expressions = 0

    for batch in loader:
        images = batch["images"].to(device, non_blocking=True)
        targets = batch["targets"].to(device, non_blocking=True)

        if targets.size(1) < 2:
            continue

        decoder_inputs = targets[:, :-1]
        decoder_targets = targets[:, 1:]

        logits = model(
            images,
            decoder_inputs,
        )

        loss = criterion(
            logits.reshape(-1, VOCAB_SIZE),
            decoder_targets.reshape(-1),
        )

        valid_token_count = (
            decoder_targets != PAD_ID
        ).sum().item()

        running_loss += loss.item() * max(valid_token_count, 1)
        total_tokens += valid_token_count

        correct_expressions += batch_exact_match(
            logits,
            decoder_targets,
        )
        total_expressions += targets.size(0)

    epoch_loss = (
        running_loss / total_tokens
        if total_tokens > 0
        else math.nan
    )

    exp_rate = (
        correct_expressions / total_expressions
        if total_expressions > 0
        else 0.0
    )

    return epoch_loss, exp_rate


In [ ]:

# ---------------------------------------------------------------------
# Checkpoint helpers
# ---------------------------------------------------------------------

def save_checkpoint(
    path,
    epoch,
    model,
    optimizer,
    train_loss,
    valid_loss,
    valid_exp_rate,
):
    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "train_loss": train_loss,
        "valid_loss": valid_loss,
        "valid_exp_rate": valid_exp_rate,
        "config": {
            "batch_size": BATCH_SIZE,
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "embedding_dim": EMBEDDING_DIM,
            "hidden_dim": HIDDEN_DIM,
            "num_layers": NUM_LAYERS,
            "max_target_length": MAX_TARGET_LENGTH,
            "seed": SEED,
            "vocab_size": VOCAB_SIZE,
            "pad_id": PAD_ID,
            "sos_id": SOS_ID,
            "eos_id": EOS_ID,
            "unk_id": UNK_ID,
        },
    }

    torch.save(checkpoint, path)


In [ ]:

# ---------------------------------------------------------------------
# Training loop
# ---------------------------------------------------------------------

history = {
    "epoch": [],
    "train_loss": [],
    "valid_loss": [],
    "valid_exp_rate": [],
    "epoch_time_seconds": [],
}

best_exp_rate = -1.0
best_valid_loss = float("inf")

print("=" * 80)
print("STARTING TRAINING")
print("=" * 80)

training_start = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    epoch_start = time.time()

    train_loss = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        DEVICE,
    )

    valid_loss, valid_exp_rate = evaluate(
        model,
        valid_loader,
        criterion,
        DEVICE,
    )

    epoch_time = time.time() - epoch_start

    history["epoch"].append(epoch)
    history["train_loss"].append(float(train_loss))
    history["valid_loss"].append(float(valid_loss))
    history["valid_exp_rate"].append(float(valid_exp_rate))
    history["epoch_time_seconds"].append(float(epoch_time))

    # Save the latest state every epoch.
    save_checkpoint(
        LAST_CHECKPOINT_PATH,
        epoch,
        model,
        optimizer,
        train_loss,
        valid_loss,
        valid_exp_rate,
    )

    # Primary selection criterion: validation ExpRate.
    # Validation loss is used as a tie-breaker.
    is_better = (
        valid_exp_rate > best_exp_rate
        or (
            valid_exp_rate == best_exp_rate
            and valid_loss < best_valid_loss
        )
    )

    if is_better:
        best_exp_rate = valid_exp_rate
        best_valid_loss = valid_loss

        save_checkpoint(
            CHECKPOINT_PATH,
            epoch,
            model,
            optimizer,
            train_loss,
            valid_loss,
            valid_exp_rate,
        )

        best_marker = "  <-- BEST"

    else:
        best_marker = ""

    elapsed = time.time() - training_start

    print(
        f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
        f"train loss: {train_loss:.5f} | "
        f"valid loss: {valid_loss:.5f} | "
        f"valid ExpRate: {valid_exp_rate:.4f} | "
        f"time: {epoch_time:.1f}s"
        f"{best_marker}"
    )

print()
print(f"Training complete in {elapsed / 60:.2f} minutes.")
print(f"Best validation ExpRate: {best_exp_rate:.4f}")
print(f"Best checkpoint: {CHECKPOINT_PATH}")


In [ ]:

# ---------------------------------------------------------------------
# Save training history
# ---------------------------------------------------------------------

with open(HISTORY_PATH, "w", encoding="utf-8") as f:
    json.dump(history, f, indent=2)

history_df = pd.DataFrame(history)
history_df


In [ ]:

# ---------------------------------------------------------------------
# Plot training and validation loss
# ---------------------------------------------------------------------

plt.figure(figsize=(8, 5))

plt.plot(
    history["epoch"],
    history["train_loss"],
    label="Train loss",
)

plt.plot(
    history["epoch"],
    history["valid_loss"],
    label="Validation loss",
)

plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()

loss_plot_path = TRAINING_ROOT / "loss_curve.png"
plt.savefig(loss_plot_path, dpi=150)
plt.show()

print("Saved:", loss_plot_path)


In [ ]:

# ---------------------------------------------------------------------
# Plot validation ExpRate
# ---------------------------------------------------------------------

plt.figure(figsize=(8, 5))

plt.plot(
    history["epoch"],
    history["valid_exp_rate"],
    marker="o",
    label="Validation ExpRate",
)

plt.xlabel("Epoch")
plt.ylabel("ExpRate")
plt.title("Validation Expression Exact-Match Rate")
plt.ylim(0, 1)
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()

exprate_plot_path = TRAINING_ROOT / "validation_exprate.png"
plt.savefig(exprate_plot_path, dpi=150)
plt.show()

print("Saved:", exprate_plot_path)


In [ ]:

# ---------------------------------------------------------------------
# Load the best checkpoint
# ---------------------------------------------------------------------

best_checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=DEVICE,
)

model.load_state_dict(
    best_checkpoint["model_state_dict"]
)

print("Best checkpoint loaded.")
print("Best epoch:", best_checkpoint["epoch"])
print("Best validation loss:", best_checkpoint["valid_loss"])
print("Best validation ExpRate:", best_checkpoint["valid_exp_rate"])



## Training outputs

After successful execution, the important artifacts are:

```text
outputs/
├── models/
│   ├── baseline_best.pt
│   └── baseline_last.pt
└── training/
    ├── baseline_training_history.json
    ├── loss_curve.png
    └── validation_exprate.png
```

`baseline_best.pt` is selected using validation ExpRate, with validation loss as the tie-breaker.

The next evaluation notebook can load `baseline_best.pt` and perform autoregressive inference on the test set.
